<a href="https://colab.research.google.com/github/AzaharaAED/Proyecto_ecosistema/blob/main/Sin_pip_Proyecto_modulo_general_gemini_personalizado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================
# IMPORTS
# =====================================

import os
import json
import concurrent.futures
import google.generativeai as genai

import comanda
import nevhor
import bascula


# =====================================
# CONFIGURAR GEMINI
# =====================================

def configurar_gemini(api_key=None):
    api_key = api_key or os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
    if not api_key:
        raise ValueError(
            "No se ha encontrado la API key de Gemini. "
            "Define GEMINI_API_KEY o GOOGLE_API_KEY, o pásala a la función."
        )
    genai.configure(api_key=api_key)


# =====================================
# EJECUCIÓN CON TIMEOUT
# =====================================

def ejecutar_con_timeout(func, timeout=15, *args, **kwargs):
    with concurrent.futures.ThreadPoolExecutor() as executor:
        future = executor.submit(func, *args, **kwargs)
        return future.result(timeout=timeout)


# =====================================
# TIPOS DE PETICIÓN PARA WEB
# =====================================

TIPOS_PETICION = {
    "ver_progreso": "ver_progreso",
    "que_puedo_cocinar": "que_puedo_cocinar",
    "recomendar_receta": "recomendar_receta",
    "generar_dieta": "generar_dieta",
    "otra_consulta": "otra_consulta"
}

def construir_comanda_web(tipo_peticion, texto_usuario):
    tipo_peticion = TIPOS_PETICION.get(tipo_peticion, "otra_consulta")
    return comanda.obtener_comanda_para_modelo_desde_texto(
        tipo_peticion=tipo_peticion,
        texto_usuario=texto_usuario
    )


# =====================================
# CONTEXTO GLOBAL
# =====================================

def modelo_general_personalizado(
    user_id,
    salida_bascula,
    salida_comanda,
    salida_nevera=None,
    salida_horno=None
):
    return {
        "user_id": user_id,
        "datos_bascula": salida_bascula,
        "peticion_usuario": salida_comanda,
        "nevera_actual": salida_nevera or {},
        "historial_horno_preferencias": salida_horno or {}
    }


# =====================================
# SYSTEM PROMPT
# =====================================

SYSTEM_PROMPT = """
Eres el asistente central de un ecosistema de electrodomésticos inteligentes.

Tu tarea es responder de forma útil, personalizada y coherente usando el contexto disponible.

Interpretación del contexto:
- La nevera representa alimentos disponibles actualmente en casa del usuario.
- El horno representa un histórico de platos o alimentos que el usuario suele cocinar y, por tanto, sus preferencias habituales. No representa inventario actual.

Reglas de comportamiento:
1. Si recomiendas recetas o comidas, prioriza lo que hay actualmente en la nevera.
2. Usa el histórico del horno como señal de gustos del usuario cuando ayude a personalizar mejor entre varias opciones.
3. Si el usuario pide una receta, propone varias opciones, no solo una.
4. Si el usuario pide una dieta, usa los datos disponibles de báscula y contexto para ajustar calorías y cantidades orientativas por comida.
5. No inventes datos ausentes. Si falta algo importante, dilo claramente.
6. Responde siempre en español, con tono natural, útil y accionable.
7. Termina muchas veces con una posibilidad de ajuste o feedback, para permitir conversación.
"""


# =====================================
# RESPUESTA NORMAL CON GEMINI
# =====================================

def responder_con_gemini(contexto, mensaje_usuario, api_key=None, model_name="gemini-2.5-flash"):
    configurar_gemini(api_key)

    if not isinstance(contexto, dict):
        return "Error: el contexto no tiene un formato válido."

    prompt = f"""
MENSAJE DEL USUARIO:
{mensaje_usuario}

CONTEXTO ESTRUCTURADO:
{json.dumps(contexto, ensure_ascii=False, indent=2)}

Genera una única respuesta final para el usuario.
"""

    modelo = genai.GenerativeModel(
        model_name=model_name,
        system_instruction=SYSTEM_PROMPT
    )

    try:
        response = ejecutar_con_timeout(
            modelo.generate_content,
            timeout=15,
            prompt=prompt,
            generation_config={
                "temperature": 0.2,
                "max_output_tokens": 3000
            }
        )
        return getattr(response, "text", str(response))

    except concurrent.futures.TimeoutError:
        return "El asistente tardó demasiado en responder. Inténtalo de nuevo."

    except Exception as e:
        mensaje_error = str(e)

        if "429" in mensaje_error or "RESOURCE_EXHAUSTED" in mensaje_error or "quota" in mensaje_error.lower():
            return (
                "Gemini no ha podido responder porque la API key o el proyecto "
                "no tienen cuota disponible ahora mismo."
            )

        return f"Error al llamar a Gemini: {mensaje_error}"


# =====================================
# DIETA EN TEXTO CON GEMINI
# =====================================

def generar_dieta_texto_con_gemini(contexto, mensaje_usuario, api_key=None, model_name="gemini-2.5-flash"):
    configurar_gemini(api_key)

    prompt = f"""
Eres un asistente nutricional dentro de un ecosistema de electrodomésticos inteligentes.

Genera una dieta personalizada en español, clara, útil y práctica.

CONTEXTO DEL USUARIO:
- User ID: {contexto.get("user_id")}
- Datos de báscula: {contexto.get("datos_bascula")}
- Preferencias (histórico horno): {contexto.get("historial_horno_preferencias")}
- Ingredientes disponibles: {contexto.get("nevera_actual")}

PETICIÓN DEL USUARIO:
{mensaje_usuario}

1. Calorías diarias estimadas
2. Explicación breve
3. Día completo de dieta:
   - Desayuno
   - Media mañana
   - Comida
   - Merienda
   - Cena
4. Alternativas opcionales
5. Pregunta final de ajuste
"""

    modelo = genai.GenerativeModel(
        model_name=model_name,
        system_instruction=SYSTEM_PROMPT
    )

    try:
        response = ejecutar_con_timeout(
            modelo.generate_content,
            timeout=20,
            prompt=prompt,
            generation_config={
                "temperature": 0.1,
                "max_output_tokens": 3000
            }
        )
        return getattr(response, "text", str(response))

    except concurrent.futures.TimeoutError:
        return "El asistente tardó demasiado en generar la dieta. Inténtalo de nuevo."

    except Exception as e:
        mensaje_error = str(e)

        if "429" in mensaje_error or "RESOURCE_EXHAUSTED" in mensaje_error or "quota" in mensaje_error.lower():
            return (
                "Gemini no ha podido generar la dieta porque la API key o el proyecto "
                "no tienen cuota disponible ahora mismo."
            )

        return f"Error al generar la dieta: {mensaje_error}"


# =====================================
# FUNCIÓN PRINCIPAL PARA WEB
# =====================================

def ejecutar_asistente_personalizado_web(
    user_id,
    tipo_peticion,
    texto_usuario,
    bascula_df,
    api_key=None,
    model_name="gemini-2.5-flash",
    url_nevera=None,
    url_horno=None
):
    if not user_id:
        return {"estado": "error", "respuesta": "Falta user_id."}

    if not texto_usuario:
        return {"estado": "error", "respuesta": "Falta texto_usuario."}

    salida_comanda = construir_comanda_web(
        tipo_peticion=tipo_peticion,
        texto_usuario=texto_usuario
    )

    salida_nevera = {}
    if url_nevera:
        try:
            salida_nevera = nevhor.obtener_ingredientes_nevera_para_modelo(url_nevera)
        except Exception as e:
            salida_nevera = {"error": f"No se pudo procesar la nevera: {e}"}

    salida_horno = {}
    if url_horno:
        try:
            salida_horno = nevhor.obtener_plato_para_modelo(url_horno)
        except Exception as e:
            salida_horno = {"error": f"No se pudo procesar el horno: {e}"}

    salida_bascula = bascula.obtener_datos_bascula_para_modelo(
        user_id=user_id,
        bascula_df=bascula_df
    )

    contexto = modelo_general_personalizado(
        user_id=user_id,
        salida_bascula=salida_bascula,
        salida_comanda=salida_comanda,
        salida_nevera=salida_nevera,
        salida_horno=salida_horno
    )

    mensaje_usuario = salida_comanda.get("texto_usuario") or texto_usuario

    if salida_comanda.get("tipo_peticion") == "generar_dieta":
        respuesta = generar_dieta_texto_con_gemini(
            contexto=contexto,
            mensaje_usuario=mensaje_usuario,
            api_key=api_key,
            model_name=model_name
        )
    else:
        respuesta = responder_con_gemini(
            contexto=contexto,
            mensaje_usuario=mensaje_usuario,
            api_key=api_key,
            model_name=model_name
        )

    return {
        "estado": "ok",
        "user_id": user_id,
        "contexto": contexto,
        "respuesta": respuesta
    }

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


ModuleNotFoundError: No module named 'comanda'